In [ ]:
import random
import numpy as np
import tensorflow as tf
import pandas as pd
import matplotlib.pyplot as plt

from keras.layers import Dense,InputLayer,Lambda,TextVectorization,Flatten,SimpleRNN,Reshape,Embedding,Input,Flatten,LSTM,Dropout
from keras.models import Sequential,Model
from keras.optimizers import Adam
from keras.preprocessing.sequence import pad_sequences

from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer

In [ ]:
seed = 42
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

In [ ]:
samples = pd.read_csv('samples.csv')
samples = samples.sample(50000)

In [ ]:
samples['Cleaned'] = samples['Review'].str.lower()
samples['Cleaned'] = samples['Cleaned'].str.replace(r'<[^>]*>',' ',regex=True)
samples['Cleaned'] = samples['Cleaned'].str.replace(r'[^a-z\s]', '', regex=True)
samples['Cleaned'] = samples['Cleaned'].str.replace(r'\s+', ' ', regex=True)

In [ ]:
train, test = train_test_split(samples[['Cleaned', 'Sentiment']], test_size=.2, random_state=seed)
train, val = train_test_split(train, test_size=.2, random_state=seed)

In [ ]:
max_tokens = 1024
output_sequence_length = 4096

In [ ]:
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
trainer = BpeTrainer(vocab_size=max_tokens, min_frequency=2)
trained = tokenizer.train_from_iterator(train['Cleaned'], trainer=trainer)

In [ ]:
max([len(tokenizer.encode(r).ids) for r in train['Cleaned']])

In [ ]:
x_train = pad_sequences([tokenizer.encode(r).ids for r in train['Cleaned']], maxlen=output_sequence_length, padding='post')
y_train = train['Sentiment'].values

x_val = pad_sequences([tokenizer.encode(r).ids for r in val['Cleaned']], maxlen=output_sequence_length, padding='post')
y_val = val['Sentiment'].values

x_test = pad_sequences([tokenizer.encode(r).ids for r in test['Cleaned']], maxlen=output_sequence_length, padding='post')
y_test = test['Sentiment'].values

In [ ]:
input = Input(shape=(output_sequence_length,))
output = Dense(1, activation='sigmoid')(input)

model = Model(inputs=input, outputs=output)

optimizer = Adam(learning_rate=.0001)
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
history = model.fit(x_train,y_train,epochs=250,batch_size=256,validation_data=(x_val,y_val)).history

In [ ]:
evaluated = model.evaluate(x_test,y_test)

In [ ]:
plt.plot(history['loss'])
plt.plot(history['val_loss'])

plt.show()

In [ ]:
plt.plot(history['accuracy'])
plt.plot(history['val_accuracy'])

plt.show()